# MixStyleGAN — Two-Style Painting Blender (Colab PoC)

Pipeline: **Stable Diffusion 1.5** + **ControlNet (Canny)** for structure preservation + **two IP-Adapters** for style conditioning, with optional **spatial masking** so each style can be applied to a different region of the image.

**Setup:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. The last cell prints a `gradio.live` URL — open it in any browser.

**Inputs:** content image + two style references (e.g., Van Gogh + Picasso) + two weight sliders + an optional paintable mask defining which region gets Style A (the rest gets Style B).

## 1. Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate peft controlnet_aux gradio safetensors

## 2. Imports and GPU check

In [ ]:
import torch
import numpy as np
import cv2
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, DDIMScheduler
from diffusers.image_processor import IPAdapterMaskProcessor
import gradio as gr

assert torch.cuda.is_available(), 'GPU not available — set Runtime > Change runtime type > T4 GPU'
device = 'cuda'
dtype = torch.float16
print(f'GPU: {torch.cuda.get_device_name()}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Load models

First run downloads ~3.5 GB of weights and caches them for the session.

**Two ControlNets stacked:**
- **Canny** preserves edges/structure from the input.
- **Tile** carries the input's *colors and composition* through to the result, so the original palette persists even under heavy IP-Adapter styling.

**Two IP-Adapters loaded** (base, not Plus) for spatial style mixing via `cross_attention_kwargs={'ip_adapter_masks': [mask_a, mask_b]}`.

In [ ]:
controlnet_canny = ControlNetModel.from_pretrained(
    'lllyasviel/sd-controlnet-canny',
    torch_dtype=dtype,
)

controlnet_tile = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11f1e_sd15_tile',
    torch_dtype=dtype,
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    controlnet=[controlnet_canny, controlnet_tile],
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
).to(device)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)

pipe.load_ip_adapter(
    'h94/IP-Adapter',
    subfolder=['models', 'models'],
    weight_name=['ip-adapter_sd15.safetensors', 'ip-adapter_sd15.safetensors'],
)

mask_processor = IPAdapterMaskProcessor()
pipe.set_progress_bar_config(disable=True)
print('Loaded: SD 1.5 + ControlNet (Canny + Tile) + 2x IP-Adapter (base)')

## 4. Spatial style mixing

Two modes:

1. **Global mix** (no mask drawn) — both styles apply to the entire image at their slider weights, like before.
2. **Spatial mix** (mask drawn) — Style A applies only inside the painted region, Style B applies to the inverse. Slider weights still control intensity within each region.

The mask is extracted from the painted layer of an `ImageEditor` widget. White (painted) = Style A region; transparent (untouched) = Style B region.

In [ ]:
BASE = 512  # SD 1.5 native resolution; we keep the shorter side near this
MAX_SIDE = 768  # cap to avoid duplication artifacts on very long aspects

def target_size(image, base=BASE, max_side=MAX_SIDE, multiple=8):
    """Compute (W, H) for generation: aspect-preserving, multiples of 8, short side near `base`, long side <= max_side."""
    w, h = image.size
    short = min(w, h)
    scale = base / short
    new_w = int(round(w * scale / multiple) * multiple)
    new_h = int(round(h * scale / multiple) * multiple)
    longer = max(new_w, new_h)
    if longer > max_side:
        cap = max_side / longer
        new_w = max(multiple, int(round(new_w * cap / multiple) * multiple))
        new_h = max(multiple, int(round(new_h * cap / multiple) * multiple))
    return new_w, new_h

def make_canny(image, low=100, high=200):
    arr = np.array(image)
    edges = cv2.Canny(arr, int(low), int(high))
    return Image.fromarray(np.stack([edges] * 3, axis=-1))

def extract_mask_from_editor(editor_value, target_w, target_h):
    """Pull a binary mask from a gr.ImageEditor value, resized to (target_w, target_h).
    Returns (mask_a_pil, mask_b_pil) or (None, None) if nothing was painted."""
    if editor_value is None:
        return None, None
    layers = editor_value.get('layers', [])
    if not layers:
        return None, None
    layer = layers[0]
    if layer is None:
        return None, None
    layer = layer.convert('RGBA').resize((target_w, target_h))
    alpha = np.array(layer)[..., 3]
    if alpha.max() == 0:
        return None, None
    mask_a = (alpha > 0).astype(np.uint8) * 255
    mask_b = 255 - mask_a
    return Image.fromarray(mask_a), Image.fromarray(mask_b)

def full_mask(target_w, target_h):
    return Image.fromarray(np.full((target_h, target_w), 255, dtype=np.uint8))

def preview_canny(content_image, low, high):
    if content_image is None:
        return None
    W, H = target_size(content_image)
    content = content_image.convert('RGB').resize((W, H))
    return make_canny(content, int(low), int(high))

def generate(content_image, mask_editor, style_a, style_b, weight_a, weight_b, prompt, steps, controlnet_scale, guidance, canny_low, canny_high, tile_scale, seed):
    if content_image is None or style_a is None or style_b is None:
        raise gr.Error('Upload a content image and both style references.')

    original_size = content_image.size  # (W, H) — full input resolution
    W, H = target_size(content_image)

    content = content_image.convert('RGB').resize((W, H))
    canny = make_canny(content, canny_low, canny_high)

    mask_a, mask_b = extract_mask_from_editor(mask_editor, W, H)
    spatial_mode = mask_a is not None
    if not spatial_mode:
        mask_a = mask_b = full_mask(W, H)

    masks_tensor = mask_processor.preprocess([mask_a, mask_b], height=H, width=W)
    ip_adapter_masks = [masks_tensor[i:i+1] for i in range(2)]

    pipe.set_ip_adapter_scale([float(weight_a), float(weight_b)])
    generator = torch.Generator(device=device).manual_seed(int(seed))

    result = pipe(
        prompt=prompt or 'a painting',
        image=[canny, content],
        ip_adapter_image=[style_a.convert('RGB'), style_b.convert('RGB')],
        cross_attention_kwargs={'ip_adapter_masks': ip_adapter_masks},
        num_inference_steps=int(steps),
        controlnet_conditioning_scale=[float(controlnet_scale), float(tile_scale)],
        guidance_scale=float(guidance),
        height=H,
        width=W,
        generator=generator,
    ).images[0]

    # Resize result back to the input's exact dimensions so output matches input pixel-for-pixel
    result = result.resize(original_size, Image.LANCZOS)

    label = 'spatial' if spatial_mode else 'global'
    info = f'**Mode:** {label}  •  **Generated at:** {W}x{H}  •  **Output at:** {original_size[0]}x{original_size[1]}'
    return result, canny, mask_a, info

## 5. Gradio UI

Drop your content image into the **Content image** input — it auto-loads into the mask editor as background. Paint over the region where you want Style A to apply; the rest of the image gets Style B. Leave the mask untouched to fall back to global mixing.

In [ ]:
with gr.Blocks(title='MixStyleGAN — 2-Style Painting Blender') as demo:
    gr.Markdown('## MixStyleGAN — Mix Two Painting Styles')

    with gr.Row():
        with gr.Column(scale=3):
            with gr.Row():
                content_in = gr.Image(label='Content', type='pil', height=220)
                mask_editor = gr.ImageEditor(
                    label='Paint Style A region (transparent = Style B)',
                    type='pil',
                    sources=['upload'],
                    transforms=[],
                    layers=False,
                    height=260,
                    brush=gr.Brush(colors=['#FFFFFF'], color_mode='fixed', default_size=40),
                )

            with gr.Row():
                style_a_in = gr.Image(label='Style A', type='pil', height=160)
                style_b_in = gr.Image(label='Style B', type='pil', height=160)

            with gr.Row():
                weight_a = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Style A weight')
                weight_b = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Style B weight')

            tile_scale = gr.Slider(0.0, 1.0, value=0.4, step=0.05,
                                    label='Color preservation (Tile)',
                                    info='Pulls original image colors through. 0 = full style takeover; 0.3–0.5 = sweet spot; 0.8+ = mostly original palette.')

            run = gr.Button('Generate', variant='primary', size='lg')

            with gr.Accordion('Edge sensitivity (Canny)', open=False):
                with gr.Row():
                    canny_low = gr.Slider(0, 255, value=100, step=5, label='Low threshold')
                    canny_high = gr.Slider(0, 255, value=200, step=5, label='High threshold')

            with gr.Accordion('Advanced', open=False):
                prompt = gr.Textbox(label='Prompt (optional)', value='a painting', lines=1)
                with gr.Row():
                    steps = gr.Slider(15, 50, value=30, step=1, label='Steps')
                    controlnet_scale = gr.Slider(0.0, 1.5, value=1.0, step=0.05, label='Canny scale')
                    guidance = gr.Slider(1.0, 12.0, value=7.0, step=0.5, label='Guidance')
                seed = gr.Number(label='Seed', value=42, precision=0)

        with gr.Column(scale=2):
            output = gr.Image(label='Result', type='pil', height=480)
            mode_label = gr.Markdown()
            with gr.Row():
                mask_view = gr.Image(label='Mask (debug)', type='pil', height=180)
                canny_view = gr.Image(label='Canny (live)', type='pil', height=180)

    content_in.change(lambda img: img, content_in, mask_editor)

    for trigger in [content_in, canny_low, canny_high]:
        trigger.change(
            preview_canny,
            [content_in, canny_low, canny_high],
            canny_view,
        )

    run.click(
        generate,
        [content_in, mask_editor, style_a_in, style_b_in, weight_a, weight_b, prompt, steps, controlnet_scale, guidance, canny_low, canny_high, tile_scale, seed],
        [output, canny_view, mask_view, mode_label],
    )

demo.launch(share=True, debug=True)

## Notes

- **Color preservation (Tile)** at 0.4 is a good starting point. Push to 0.6–0.8 if you want very faithful original colors, drop to 0.1–0.2 if you want only a hint of the original palette.
- **Tile and Canny work together**: Canny holds shapes, Tile holds colors and composition. You can run with high Canny + 0 Tile (current style transfer) or low Canny + high Tile (looser shapes, faithful palette).
- **Slider semantics:** weights still control intensity. With a mask drawn, weight_a controls how strongly Style A acts inside its region; weight_b controls Style B outside it.
- **Speed on T4:** ~20–30 s per 512x512 image at 30 steps with two ControlNets and two IP-Adapters loaded.